In [87]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [88]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [89]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [90]:
len(documents)

72

In [91]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [92]:
for doc in documents[:3]:
    print(doc["filename"])

01-agentic-rag/lessons/01-intro.md
01-agentic-rag/lessons/02-environment.md
01-agentic-rag/lessons/03-rag.md


In [93]:
import json

user_prompt = None

for doc in documents[:3]:
   user_prompt = json.dumps(doc)

In [94]:
user_prompt

'{"content": "# RAG\\n\\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=JktYwBIDErk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\\n\\nWe run free Zoomcamp courses at DataTalks.Club on data engineering,\\nmachine learning, MLOps, and other topics. Each course has its own\\nFAQ document with common questions and answers.\\n\\nSome of these documents have over 300 questions. Students ask us\\nthings in Slack like \\"Can I still join after the course started?\\" or\\n\\"How do I get a certificate?\\" Finding those answers in the FAQ is\\ntedious.\\n\\nWe want a bot that takes all this knowledge and answers student\\nquestions in natural language.\\n\\nIn this module, we\'ll build that system. But first, let\'s see why we\\ncan\'t send the question straight to an LLM and call it a day.\\n\\n## Plain LLMs lack our data\\n\\nFirst, let\'s define a function to talk to the LLM:\\n\\n```python\\ndef llm(prompt):\\n    response = openai_client.responses.create(\\n        model=\\"gpt-5.4-

In [95]:
from evaluation_utils import llm_structured
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['Why doesn’t a plain LLM answer course FAQ questions correctly without extra help?', 'How does adding the FAQ text into the prompt change the answer the model gives?', 'What does RAG mean in this lesson, and what are its two main parts?', 'Why is the search step so important in a RAG system?', 'Can you swap out the search engine or the LLM in this setup without changing the whole system?']


In [96]:
print(usage)

ResponseUsage(input_tokens=1753, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=98, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1851)


In [97]:
import pandas as pd

ground_truth = pd.read_csv("ground-truth.csv").to_dict(orient="records")

print(len(ground_truth))
print(ground_truth[0])

360
{'question': "What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?", 'filename': '01-agentic-rag/lessons/01-intro.md'}


In [98]:
from gitsource import chunk_documents

chunks = chunk_documents(
    documents,
    size=2000,
    step=1000
)

print(len(chunks))

295


In [99]:
print (chunks[1])
chunks[0].keys()

{'start': 1000, 'content': 'the next\nword based on what you typed so far.\n\nA large language model does the same thing, but at a much larger scale.\nIt has billions of parameters and is trained on most of the text on the\ninternet. When it predicts the next word, it feels like you\'re talking\nto an intelligent being. It understands what you ask and gives\nmeaningful answers.\n\nIn this course, we treat LLMs as black boxes. We won\'t look inside or\ncover the theory, and we won\'t host a model ourselves. We use an LLM\nprovider and call it over an API. For us, an LLM is a box: text goes in,\ntext comes out.\n\nBut LLMs have limitations:\n\n- Knowledge cutoff: they only know what was in their training data.\n  If you ask about something that happened after training, they won\'t\n  know - or worse, they\'ll make something up.\n- No access to your data: they can\'t see your documents, databases,\n  or internal systems unless you provide that information.\n- Hallucinations: they sometime

dict_keys(['start', 'content', 'filename'])

In [100]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(chunks)

In [137]:
def text_search(query, num_results=5):
    results = index.search(
        query=query,
        num_results=num_results
    )
    return results

In [102]:
text_search("what is RAG?")

[{'start': 3000,
  'content': 'we drop it.\n\nBuild a prompt that includes both the question and the context:\n\n```python\nprompt = f"""\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n\nQuestion:\n{question}\n\nContext:\n{context}\n"""\n```\n\nInstead of sending the raw question to the LLM, we send this prompt:\n\n```python\nanswer = llm(prompt)\nprint(answer)\n```\n\nAfter that, the answer is correct: "Yes, you can still join. If you want to\nreceive a certificate, you need to submit your project while\nsubmissions are still open."\n\nThis is the answer we actually want to give to our students. What we\njust did is nothing but RAG.\n\n## Retrieval plus generation\n\nRAG stands for Retrieval-Augmented Generation. Generation is the LLM\nproducing text, and retrieval is search. We retriev

In [103]:
from embedder import Embedder

embed = Embedder()


In [104]:
Y = []

for chunk in chunks:
    text = chunk["content"]
    vector = embed.encode(text)
    Y.append(vector)

import numpy as np
Y = np.array(Y)

print("Y shape:", Y.shape)

Y shape: (295, 384)


In [105]:
from minsearch import VectorSearch

vindex = VectorSearch(
    keyword_fields=["filename"]
)

vindex.fit(Y, chunks)


In [138]:
def vector_search(query, num_results=5):
    query_vector = embed.encode(query)

    results = vindex.search(
        query_vector,
        num_results=num_results
    )

    return results

In [107]:
vector_search("What is RAG?")

[{'start': 0,
  'content': '# Agent Evaluation\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=2SW86BehVdI&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nFor RAG, we used the A->Q->A\' setup:\n\n- A = original answer in the FAQ\n- Q = generated question from this answer\n- A\' = answer produced by our RAG system\n\nFor agents, we use the same setup. A\' comes from an agent instead of a\nfixed RAG pipeline.\n\nWe also save the trajectory. Here, the trajectory means only the tool\ncalls the agent made before producing the final answer.\n\n## Loading the data\n\nUse the same ground truth questions:\n\n```python\nimport pandas as pd\n\ndf_ground_truth = pd.read_csv("data/ground_truth-new.csv")\nground_truth = df_ground_truth.to_dict(orient="records")\n```\n\nLoad the FAQ documents and the search index:\n\n```python\nfrom ingest import load_faq_data, build_index\n\ndocuments = load_faq_data()\n\ndocuments_llm = []\n\nfor doc in documents:\n    if doc["course"] == "llm-zoomcamp":\

In [108]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [109]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [110]:
q = ground_truth[0]["question"]
q
ground_truth[0]

{'question': "What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?",
 'filename': '01-agentic-rag/lessons/01-intro.md'}

In [111]:
text_search(q)

[{'start': 3000,
  'content': 'we drop it.\n\nBuild a prompt that includes both the question and the context:\n\n```python\nprompt = f"""\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n\nQuestion:\n{question}\n\nContext:\n{context}\n"""\n```\n\nInstead of sending the raw question to the LLM, we send this prompt:\n\n```python\nanswer = llm(prompt)\nprint(answer)\n```\n\nAfter that, the answer is correct: "Yes, you can still join. If you want to\nreceive a certificate, you need to submit your project while\nsubmissions are still open."\n\nThis is the answer we actually want to give to our students. What we\njust did is nothing but RAG.\n\n## Retrieval plus generation\n\nRAG stands for Retrieval-Augmented Generation. Generation is the LLM\nproducing text, and retrieval is search. We retriev

In [112]:
vector_search(q)

[{'start': 0,
  'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour ph

In [113]:
def compute_relevance_text(q):
    filename = q["filename"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == filename))

    return relevance

In [114]:
q = ground_truth[0]
print(q["question"])
compute_relevance_text(q)


What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?


[0, 0, 0, 0, 1]

In [115]:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [116]:
ground_truth_sample = ground_truth[:15]
relevance_total_text = compute_relevance_total_text(ground_truth_sample)

  0%|          | 0/15 [00:00<?, ?it/s]

In [117]:
relevance_total_text

[[0, 0, 0, 0, 1],
 [1, 0, 1, 0, 0],
 [1, 1, 0, 0, 1],
 [1, 0, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 1, 1, 0, 0],
 [1, 1, 0, 0, 0],
 [0, 1, 0, 0, 1],
 [0, 0, 0, 0, 0],
 [0, 1, 1, 0, 0],
 [0, 0, 0, 0, 0]]

In [118]:
def compute_relevance(q, search_function):
    filename = q["filename"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == filename))

    return relevance

In [119]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [120]:
relevance_total = compute_relevance_total(ground_truth_sample, text_search)
relevance_total

  0%|          | 0/15 [00:00<?, ?it/s]

[[0, 0, 0, 0, 1],
 [1, 0, 1, 0, 0],
 [1, 1, 0, 0, 1],
 [1, 0, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 1, 1, 0, 0],
 [1, 1, 0, 0, 0],
 [0, 1, 0, 0, 1],
 [0, 0, 0, 0, 0],
 [0, 1, 1, 0, 0],
 [0, 0, 0, 0, 0]]

In [121]:
relevance_total = compute_relevance_total(ground_truth_sample, vector_search)
relevance_total

  0%|          | 0/15 [00:00<?, ?it/s]

[[1, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 1, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 1, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [0, 1, 1, 1, 0],
 [0, 0, 1, 1, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0]]

In [122]:
relevance_total_text = compute_relevance_total(ground_truth, text_search)
relevance_total_text

  0%|          | 0/360 [00:00<?, ?it/s]

[[0, 0, 0, 0, 1],
 [1, 0, 1, 0, 0],
 [1, 1, 0, 0, 1],
 [1, 0, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 1, 1, 0, 0],
 [1, 1, 0, 0, 0],
 [0, 1, 0, 0, 1],
 [0, 0, 0, 0, 0],
 [0, 1, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 1, 1, 0, 0],
 [0, 0, 1, 1, 0],
 [0, 1, 1, 0, 0],
 [1, 1, 0, 0, 1],
 [1, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 1, 0],
 [0, 0, 0, 0, 0],
 [1, 1, 0, 0, 0],
 [1, 0, 1, 1, 0],
 [1, 0, 0, 0, 0],
 [1, 1, 0, 0, 1],
 [0, 0, 0, 0, 0],
 [0, 1, 1, 1, 0],
 [1, 0, 1, 1, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 1],
 [1, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 1, 0, 0, 0],
 [1, 1, 1, 1, 0],
 [1, 1, 1, 1, 0],
 [1, 1, 1, 1, 1],
 [1, 1, 1, 1, 1],
 [0, 0, 0, 1, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 1, 0],
 [1, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0,

In [123]:
relevance_total_vector = compute_relevance_total(ground_truth, vector_search)
relevance_total_vector

  0%|          | 0/360 [00:00<?, ?it/s]

[[1, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 1, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 1, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [0, 1, 1, 1, 0],
 [0, 0, 1, 1, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 1, 0, 1, 0],
 [1, 0, 0, 0, 0],
 [1, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 1, 1, 1],
 [1, 1, 1, 0, 1],
 [1, 0, 1, 1, 0],
 [1, 0, 0, 1, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 1, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 1],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 1, 0],
 [1, 0, 0, 1, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 1, 0, 1, 0],
 [0, 0, 1, 1, 1],
 [1, 1, 1, 0, 1],
 [1, 1, 1, 1, 0],
 [1, 1, 0, 1, 1],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0,

In [125]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [126]:
hit_rate(relevance_total_text)
# 0.933

0.7583333333333333

In [128]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [129]:
mrr(relevance_total_vector)
# 0.822

0.5486111111111112

In [139]:
def hybrid_search_1(query, k=1):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [133]:
def hybrid_search_50(query, k=50):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [134]:
def hybrid_search_100(query, k=100):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [135]:
def hybrid_search_200(query, k=200):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [142]:
relevance_total_1 = compute_relevance_total(ground_truth, hybrid_search_1)
#relevance_total_1
mrr(relevance_total_1)

  0%|          | 0/360 [00:00<?, ?it/s]

0.6481944444444449

In [143]:
relevance_total_50 = compute_relevance_total(ground_truth, hybrid_search_50)
#relevance_total_50
mrr(relevance_total_50)

  0%|          | 0/360 [00:00<?, ?it/s]

0.637916666666667

In [144]:
relevance_total_100 = compute_relevance_total(ground_truth, hybrid_search_100)
#relevance_total_100
mrr(relevance_total_100)

  0%|          | 0/360 [00:00<?, ?it/s]

0.637916666666667

In [145]:
relevance_total_200 = compute_relevance_total(ground_truth, hybrid_search_200)
#relevance_total_200
mrr(relevance_total_200)

  0%|          | 0/360 [00:00<?, ?it/s]

0.637916666666667